In [ ]:
from pathlib import Path

dataset_folder = Path(
    "/kaggle/input/competitions/"
    "enveda-CASMI26-molecule-id-mass-spectra"
)

train_path = dataset_folder / "train.parquet"
test_path = dataset_folder / "test.parquet"
submission_path = dataset_folder / "sample_submission.csv"

print("Train file exists:", train_path.exists())
print("Test file exists:", test_path.exists())
print("Sample submission exists:", submission_path.exists())

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

test_file = pq.ParquetFile(test_path)
sample_submission = pd.read_csv(submission_path)

print("Test spectra:", test_file.metadata.num_rows)
print("Test columns:", test_file.schema_arrow.names)
print("Submission columns:", sample_submission.columns.tolist())
print("Sample submission rows:", len(sample_submission))

In [ ]:
# CASMI 2026 - Prepare test spectra for exact matching

import hashlib
import numpy as np
from collections import defaultdict

test_columns = [
    "molecule_id",
    "spectrum_id",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

test_spectra = test_file.read(
    columns=test_columns
).to_pandas()


def spectrum_fingerprint(row):
    mz = np.asarray(row.ms2_mzs, dtype=np.float64)

    intensity = np.asarray(
        row.ms2_normalized_intensities,
        dtype=np.float64
    )

    fingerprint = hashlib.blake2b(
        mz.tobytes() + intensity.tobytes(),
        digest_size=16
    ).digest()

    return (row.adduct, len(mz), fingerprint)


# Connect each fingerprint to its test molecule and spectrum
test_lookup = defaultdict(list)

for row in test_spectra.itertuples(index=False):
    fingerprint = spectrum_fingerprint(row)

    test_lookup[fingerprint].append(
        (row.molecule_id, row.spectrum_id)
    )


print("CASMI 2026 - Test Spectra Prepared")
print("----------------------------------")
print("Test spectra:", len(test_spectra))
print(
    "Unique test molecules:",
    test_spectra["molecule_id"].nunique()
)
print("Unique spectrum fingerprints:", len(test_lookup))

print(
    "Submission molecule IDs match test data:",
    set(sample_submission["molecule_id"])
    == set(test_spectra["molecule_id"])
)

print("\nTest spectra prepared successfully!")

In [ ]:
# CASMI 2026 - Find exact matches in the training dataset

import pyarrow.parquet as pq
from collections import defaultdict

train_file = pq.ParquetFile(train_path)

train_columns = [
    "inchikey14",
    "normalized_smiles",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

# Skip spectra with adduct/peak-count combinations
# that do not occur in the test data.
test_combinations = {
    (row.adduct, len(row.ms2_mzs))
    for row in test_spectra.itertuples(index=False)
}

# molecule_id -> (inchikey14, SMILES) -> matched spectrum IDs
exact_hits = defaultdict(lambda: defaultdict(set))

print("CASMI 2026 - Exact Match Search")
print("-------------------------------")

for group_index in range(train_file.num_row_groups):

    for batch in train_file.iter_batches(
        row_groups=[group_index],
        batch_size=10000,
        columns=train_columns
    ):
        batch_df = batch.to_pandas()

        for row in batch_df.itertuples(index=False):

            combination = (
                row.adduct,
                len(row.ms2_mzs)
            )

            if combination not in test_combinations:
                continue

            fingerprint = spectrum_fingerprint(row)

            for molecule_id, spectrum_id in test_lookup.get(
                fingerprint, []
            ):
                exact_hits[molecule_id][
                    (row.inchikey14, row.normalized_smiles)
                ].add(spectrum_id)

    if (
        (group_index + 1) % 5 == 0
        or group_index + 1 == train_file.num_row_groups
    ):
        print(
            f"Row groups checked: {group_index + 1}/"
            f"{train_file.num_row_groups}"
        )


# Summarize without assuming that all test molecules matched
test_molecule_count = test_spectra["molecule_id"].nunique()

matched_molecules = len(exact_hits)

unique_matches = sum(
    len(structures) == 1
    for structures in exact_hits.values()
)

ambiguous_matches = sum(
    len(structures) > 1
    for structures in exact_hits.values()
)

print("\nExact matching results:")
print("Test molecules:", test_molecule_count)
print("Molecules with exact matches:", matched_molecules)
print("Molecules with one matched structure:", unique_matches)
print("Molecules with multiple matched structures:", ambiguous_matches)
print(
    "Molecules without an exact match:",
    test_molecule_count - matched_molecules
)

In [ ]:
# CASMI 2026 - Prepare exact-match predictions

exact_predictions = {}
unresolved_molecules = []

for molecule_id in sample_submission["molecule_id"]:

    matched_structures = exact_hits.get(molecule_id, {})

    # Use an exact match only when it identifies one structure
    if len(matched_structures) == 1:

        inchikey14, smiles = next(
            iter(matched_structures.keys())
        )

        if (
            isinstance(smiles, str)
            and smiles.strip()
            and ";" not in smiles
        ):
            exact_predictions[molecule_id] = smiles
        else:
            unresolved_molecules.append(molecule_id)

    else:
        unresolved_molecules.append(molecule_id)


print("CASMI 2026 - Exact Predictions")
print("------------------------------")
print("Submission molecules:", len(sample_submission))
print("Exact predictions:", len(exact_predictions))
print("Unresolved molecules:", len(unresolved_molecules))

print("\nFirst 3 exact predictions:")
for molecule_id, smiles in list(exact_predictions.items())[:3]:
    print(molecule_id, "->", smiles)

print("\nExact prediction mapping prepared!")

In [ ]:
# CASMI 2026 - Build a fallback candidate library

import pandas as pd

library_columns = [
    "inchikey14",
    "normalized_smiles",
    "molecular_formula"
]

seen_keys = set()
library_parts = []

for batch in train_file.iter_batches(
    batch_size=20000,
    columns=library_columns
):
    batch_df = batch.to_pandas()

    # Remove incomplete rows and repeated molecular structures
    batch_df = batch_df.dropna(
        subset=library_columns
    ).drop_duplicates(subset="inchikey14")

    new_rows = batch_df[
        ~batch_df["inchikey14"].isin(seen_keys)
    ]

    if not new_rows.empty:
        library_parts.append(new_rows)
        seen_keys.update(new_rows["inchikey14"])

full_candidate_library = pd.concat(
    library_parts,
    ignore_index=True
)

print("CASMI 2026 - Fallback Candidate Library")
print("---------------------------------------")
print("Unique molecular structures:", len(full_candidate_library))
print(
    "Missing SMILES:",
    full_candidate_library["normalized_smiles"].isna().sum()
)
print(
    "Missing molecular formulas:",
    full_candidate_library["molecular_formula"].isna().sum()
)

print("\nCandidate library prepared!")

In [ ]:
# CASMI 2026 - Calculate candidate reference masses

import re
import numpy as np

# Monoisotopic atomic masses in Da
atomic_masses = {
    "C": 12.000000000,
    "H": 1.007825032,
    "N": 14.003074004,
    "O": 15.994914620,
    "S": 31.972071174,
    "P": 30.973761998,
    "F": 18.998403163,
    "Cl": 34.968852682,
    "Br": 78.918337600,
    "I": 126.904468000,
    "B": 11.009305360
}


def formula_to_reference_mass(formula):
    if not isinstance(formula, str):
        return np.nan

    # Remove trailing charge notation, where present
    elemental_formula = re.sub(r"\+\d*$", "", formula)

    parts = re.findall(
        r"([A-Z][a-z]?)(\d*)",
        elemental_formula
    )

    # Reject formulas we cannot parse completely
    reconstructed = "".join(
        element + count
        for element, count in parts
    )

    if reconstructed != elemental_formula or not parts:
        return np.nan

    if any(
        element not in atomic_masses
        for element, _ in parts
    ):
        return np.nan

    return sum(
        atomic_masses[element] * int(count or 1)
        for element, count in parts
    )


full_candidate_library["reference_mass"] = (
    full_candidate_library["molecular_formula"]
    .apply(formula_to_reference_mass)
)

print("CASMI 2026 - Reference Mass Calculation")
print("---------------------------------------")
print("Total candidates:", len(full_candidate_library))
print(
    "Candidates with reference mass:",
    int(full_candidate_library["reference_mass"].notna().sum())
)
print(
    "Candidates with missing reference mass:",
    int(full_candidate_library["reference_mass"].isna().sum())
)

print("\nReference mass calculation completed!")

In [ ]:
# CASMI 2026 - Calculate neutral masses for test molecules

import pandas as pd

adduct_shifts = {
    "[M+H]+": 1.007276466621,
    "[M-H]-": -1.007276466621,
    "[M+CH2O2-H]-": 44.99820284,
    "[M+Na]+": 22.989218,
    "[M+NH4]+": 18.033823,
    "[M+K]+": 38.963158,
    "[M+Cl]-": 34.969401
}

# Read the test metadata available during this notebook run
mass_data = test_file.read(
    columns=[
        "molecule_id",
        "spectrum_id",
        "adduct",
        "precursor_mz"
    ]
).to_pandas()

mass_data["adduct_shift"] = (
    mass_data["adduct"].map(adduct_shifts)
)

mass_data["neutral_mass"] = (
    pd.to_numeric(mass_data["precursor_mz"], errors="coerce")
    - mass_data["adduct_shift"]
)

# Calculate one median neutral mass per molecule
test_molecules = (
    mass_data.groupby("molecule_id", as_index=False)
    .agg(
        median_neutral_mass=("neutral_mass", "median"),
        num_spectra=("spectrum_id", "count")
    )
)

print("CASMI 2026 - Test Neutral Masses")
print("--------------------------------")
print("Test spectra:", len(mass_data))
print("Test molecules:", len(test_molecules))

print(
    "Spectra with unknown adducts:",
    int(mass_data["adduct_shift"].isna().sum())
)

print(
    "Molecules with missing neutral mass:",
    int(test_molecules["median_neutral_mass"].isna().sum())
)

print("\nFirst 5 molecules:")
print(test_molecules.head().to_string(index=False))

In [ ]:
# CASMI 2026 - Generate submission.csv

import numpy as np
import pandas as pd
from pathlib import Path

# Prepare the mass-based candidate library
mass_library = (
    full_candidate_library
    .dropna(subset=["reference_mass", "normalized_smiles"])
    .sort_values("reference_mass")
    .reset_index(drop=True)
)

sorted_masses = mass_library["reference_mass"].to_numpy(
    dtype=float
)
sorted_smiles = mass_library["normalized_smiles"].to_numpy()

mass_lookup = test_molecules.set_index(
    "molecule_id"
)["median_neutral_mass"]

submission_rows = []

for molecule_id in sample_submission["molecule_id"]:

    selected_smiles = []
    seen_smiles = set()

    # Place an unambiguous exact match first, if available
    exact_smiles = exact_predictions.get(molecule_id)

    if exact_smiles is not None:
        selected_smiles.append(exact_smiles)
        seen_smiles.add(exact_smiles)

    # Fill remaining positions using nearby reference masses
    query_mass = float(mass_lookup.loc[molecule_id])

    if not np.isfinite(query_mass):
        raise ValueError(
            f"No usable neutral mass for {molecule_id}"
        )

    right = int(np.searchsorted(sorted_masses, query_mass))
    left = right - 1

    while len(selected_smiles) < 25:

        if left < 0:
            candidate_index = right
            right += 1

        elif right >= len(sorted_masses):
            candidate_index = left
            left -= 1

        else:
            left_error = abs(
                sorted_masses[left] - query_mass
            )
            right_error = abs(
                sorted_masses[right] - query_mass
            )

            if left_error <= right_error:
                candidate_index = left
                left -= 1
            else:
                candidate_index = right
                right += 1

        candidate_smiles = sorted_smiles[candidate_index]

        if candidate_smiles not in seen_smiles:
            selected_smiles.append(candidate_smiles)
            seen_smiles.add(candidate_smiles)

    submission_rows.append({
        "molecule_id": molecule_id,
        "smiles": ";".join(selected_smiles)
    })

submission = pd.DataFrame(submission_rows)

# Check the required format
assert submission.columns.tolist() == ["molecule_id", "smiles"]
assert len(submission) == len(sample_submission)

assert (
    submission["molecule_id"].tolist()
    == sample_submission["molecule_id"].tolist()
)

assert all(
    len(smiles.split(";")) == 25
    and len(set(smiles.split(";"))) == 25
    for smiles in submission["smiles"]
)

# Save using the filename required by Kaggle
output_path = Path("/kaggle/working/submission.csv")
submission.to_csv(output_path, index=False)

print("CASMI 2026 - Submission Created")
print("--------------------------------")
print("File exists:", output_path.exists())
print("File path:", output_path)
print("Submission rows:", len(submission))
print("SMILES per molecule: 25")
print(
    "Molecules with exact prediction:",
    sum(
        molecule_id in exact_predictions
        for molecule_id in submission["molecule_id"]
    )
)
print(
    "Molecules using mass-only fallback:",
    sum(
        molecule_id not in exact_predictions
        for molecule_id in submission["molecule_id"]
    )
)
print("\nSubmission format validation passed!")

In [10]:
# CASMI 2026 - Evaluate the current mass-only fallback

import numpy as np

# Use the variables already created in our submission notebook:
# sorted_masses, sorted_smiles, mass_lookup, exact_predictions

total = 0
top1_matches = 0
top25_matches = 0
reciprocal_rank_sum = 0.0

for molecule_id in sample_submission["molecule_id"]:

    known_smiles = exact_predictions.get(molecule_id)

    if known_smiles is None:
        continue

    query_mass = float(mass_lookup.loc[molecule_id])

    if not np.isfinite(query_mass):
        continue

    # Recreate the mass-only ranking without using the exact match
    selected_smiles = []
    seen_smiles = set()

    right = int(np.searchsorted(sorted_masses, query_mass))
    left = right - 1

    while len(selected_smiles) < 25:

        if left < 0:
            candidate_index = right
            right += 1

        elif right >= len(sorted_masses):
            candidate_index = left
            left -= 1

        else:
            left_error = abs(sorted_masses[left] - query_mass)
            right_error = abs(sorted_masses[right] - query_mass)

            if left_error <= right_error:
                candidate_index = left
                left -= 1
            else:
                candidate_index = right
                right += 1

        candidate_smiles = sorted_smiles[candidate_index]

        if candidate_smiles not in seen_smiles:
            selected_smiles.append(candidate_smiles)
            seen_smiles.add(candidate_smiles)

    total += 1

    if known_smiles in selected_smiles:
        rank = selected_smiles.index(known_smiles) + 1
        top25_matches += 1
        reciprocal_rank_sum += 1.0 / rank

        if rank == 1:
            top1_matches += 1

print("CASMI 2026 - Mass-Only Fallback Diagnostic")
print("------------------------------------------")
print("Molecules evaluated:", total)
print("Exact-match SMILES at rank 1:", top1_matches)
print("Exact-match SMILES within top 25:", top25_matches)

if total:
    print("Top-1 rate:", round(top1_matches / total, 4))
    print("Top-25 rate:", round(top25_matches / total, 4))
    print("Mean Reciprocal Rank @ 25:", round(
        reciprocal_rank_sum / total, 4
    ))

CASMI 2026 - Mass-Only Fallback Diagnostic
------------------------------------------
Molecules evaluated: 400
Exact-match SMILES at rank 1: 23
Exact-match SMILES within top 25: 221
Top-1 rate: 0.0575
Top-25 rate: 0.5525
Mean Reciprocal Rank @ 25: 0.1386


In [11]:
# CASMI 2026 - Diagnose mass-only fallback misses

import numpy as np

# Find reference masses associated with each SMILES
masses_by_smiles = {}

for row in full_candidate_library[
    ["normalized_smiles", "reference_mass"]
].itertuples(index=False):

    if np.isfinite(row.reference_mass):
        masses_by_smiles.setdefault(
            row.normalized_smiles, []
        ).append(float(row.reference_mass))


diagnostic = {
    "correct_smiles_not_in_library": 0,
    "correct_mass_within_20_ppm": 0,
    "correct_mass_above_20_ppm": 0,
    "missed_top25_but_mass_within_20_ppm": 0
}

examples = []

for molecule_id in sample_submission["molecule_id"]:

    true_smiles = exact_predictions[molecule_id]
    query_mass = float(mass_lookup.loc[molecule_id])

    reference_masses = masses_by_smiles.get(true_smiles)

    if not reference_masses:
        diagnostic["correct_smiles_not_in_library"] += 1
        continue

    best_mass_error_ppm = min(
        abs(reference_mass - query_mass)
        / query_mass * 1_000_000
        for reference_mass in reference_masses
    )

    if best_mass_error_ppm <= 20:
        diagnostic["correct_mass_within_20_ppm"] += 1
    else:
        diagnostic["correct_mass_above_20_ppm"] += 1

    # Is the correct SMILES in our previous mass-only top 25?
    predicted_smiles = submission.loc[
        submission["molecule_id"] == molecule_id,
        "smiles"
    ].iloc[0].split(";")

    # Remove the exact-match prediction from consideration.
    # The remaining entries are not necessarily identical to
    # the mass-only top 25, so use the mass-only search below.
    left = int(np.searchsorted(sorted_masses, query_mass)) - 1
    right = left + 1
    mass_only_top25 = []
    seen = set()

    while len(mass_only_top25) < 25:
        if left < 0:
            index = right
            right += 1
        elif right >= len(sorted_masses):
            index = left
            left -= 1
        elif abs(sorted_masses[left] - query_mass) <= abs(
            sorted_masses[right] - query_mass
        ):
            index = left
            left -= 1
        else:
            index = right
            right += 1

        smiles = sorted_smiles[index]

        if smiles not in seen:
            mass_only_top25.append(smiles)
            seen.add(smiles)

    if (
        true_smiles not in mass_only_top25
        and best_mass_error_ppm <= 20
    ):
        diagnostic["missed_top25_but_mass_within_20_ppm"] += 1

        if len(examples) < 5:
            examples.append(
                (molecule_id, round(best_mass_error_ppm, 4))
            )


print("CASMI 2026 - Mass-Only Diagnostic")
print("--------------------------------")
for label, value in diagnostic.items():
    print(f"{label}: {value}")

print("\nExamples missed from top 25 despite mass match:")
for molecule_id, error_ppm in examples:
    print(molecule_id, "| mass error:", error_ppm, "ppm")

CASMI 2026 - Mass-Only Diagnostic
--------------------------------
correct_smiles_not_in_library: 0
correct_mass_within_20_ppm: 400
correct_mass_above_20_ppm: 0
missed_top25_but_mass_within_20_ppm: 179

Examples missed from top 25 despite mass match:
m_006153 | mass error: 1.2033 ppm
m_00b5aa | mass error: 2.2235 ppm
m_0259d4 | mass error: 2.0597 ppm
m_02d188 | mass error: 0.6344 ppm
m_050bf1 | mass error: 2.161 ppm


In [12]:
# CASMI 2026 - Prepare references without exact test-spectrum copies

import numpy as np
import pandas as pd

molecule_id = "m_006153"
ppm_tolerance = 20

# Test spectra and observed adducts for this molecule
queries = test_spectra[
    test_spectra["molecule_id"] == molecule_id
].copy()

test_adducts = set(queries["adduct"])

# Identify the structure found through exact matching
true_key, true_smiles = next(
    iter(exact_hits[molecule_id].keys())
)

# Find all mass-matched candidate structures
query_mass = float(
    test_molecules.loc[
        test_molecules["molecule_id"] == molecule_id,
        "median_neutral_mass"
    ].iloc[0]
)

ppm_errors = (
    abs(full_candidate_library["reference_mass"] - query_mass)
    / query_mass
) * 1_000_000

candidate_keys = set(
    full_candidate_library.loc[
        ppm_errors <= ppm_tolerance,
        "inchikey14"
    ]
)

# Fingerprints of this molecule's test spectra
query_fingerprints = {
    spectrum_fingerprint(row)
    for row in queries.itertuples(index=False)
}

reference_parts = []
excluded_exact_copies = 0

# Read training data in batches
for batch in train_file.iter_batches(
    batch_size=20000,
    columns=[
        "inchikey14",
        "adduct",
        "ms2_mzs",
        "ms2_normalized_intensities"
    ]
):
    batch_df = batch.to_pandas()

    matches = batch_df[
        batch_df["inchikey14"].isin(candidate_keys)
        & batch_df["adduct"].isin(test_adducts)
    ].copy()

    if matches.empty:
        continue

    # Exclude reference spectra identical to any query spectrum
    is_exact_copy = matches.apply(
        lambda row: spectrum_fingerprint(row)
        in query_fingerprints,
        axis=1
    )

    excluded_exact_copies += int(is_exact_copy.sum())

    remaining = matches.loc[~is_exact_copy]

    if not remaining.empty:
        reference_parts.append(remaining)

experiment_references = (
    pd.concat(reference_parts, ignore_index=True)
    if reference_parts
    else pd.DataFrame()
)

print("CASMI 2026 - Spectral Ranking Experiment")
print("----------------------------------------")
print("Test molecule:", molecule_id)
print("Test spectra:", len(queries))
print("Mass-matched candidates:", len(candidate_keys))
print("Exact test-spectrum copies excluded:", excluded_exact_copies)
print("Remaining reference spectra:", len(experiment_references))

if not experiment_references.empty:
    print(
        "Candidates with remaining references:",
        experiment_references["inchikey14"].nunique()
    )
    print(
        "Remaining references for the known structure:",
        int(
            (
                experiment_references["inchikey14"]
                == true_key
            ).sum()
        )
    )

CASMI 2026 - Spectral Ranking Experiment
----------------------------------------
Test molecule: m_006153
Test spectra: 6
Mass-matched candidates: 257
Exact test-spectrum copies excluded: 6
Remaining reference spectra: 1376
Candidates with remaining references: 250
Remaining references for the known structure: 1


In [13]:
# CASMI 2026 - Rank candidates without exact spectrum copies

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Convert a spectrum into 1 Da intensity bins
def extract_features(row):
    features = np.zeros(1000, dtype=np.float32)

    mzs = np.asarray(row["ms2_mzs"], dtype=float)
    intensities = np.asarray(
        row["ms2_normalized_intensities"],
        dtype=float
    )

    for mz, intensity in zip(mzs, intensities):
        if np.isfinite(mz) and np.isfinite(intensity):
            index = int(mz)
            if 0 <= index < 1000:
                features[index] = max(
                    features[index], intensity
                )

    return features


# Prepare all mass-matched candidates
ranking = full_candidate_library[
    full_candidate_library["inchikey14"].isin(candidate_keys)
].copy()

ranking["mass_error_ppm"] = (
    abs(ranking["reference_mass"] - query_mass)
    / query_mass * 1_000_000
)

# Give every candidate a score for every query spectrum.
# A missing matching-adduct reference contributes zero.
query_count = len(queries)

score_sums = {
    key: 0.0 for key in candidate_keys
}

matched_query_counts = {
    key: 0 for key in candidate_keys
}

# Compare only spectra with the same adduct
for adduct, query_group in queries.groupby("adduct"):

    ref_group = experiment_references[
        experiment_references["adduct"] == adduct
    ]

    if ref_group.empty:
        continue

    X_query = np.stack(
        query_group.apply(
            extract_features, axis=1
        ).to_numpy()
    )

    X_ref = np.stack(
        ref_group.apply(
            extract_features, axis=1
        ).to_numpy()
    )

    similarity = cosine_similarity(X_query, X_ref)
    ref_keys = ref_group["inchikey14"].to_numpy()

    for key in np.unique(ref_keys):

        positions = np.flatnonzero(ref_keys == key)

        # Best reference match for each query spectrum
        best_scores = similarity[:, positions].max(axis=1)

        score_sums[key] += float(best_scores.sum())
        matched_query_counts[key] += len(best_scores)


ranking["spectral_score"] = ranking["inchikey14"].map(
    lambda key: (
        score_sums[key] / query_count
        if matched_query_counts[key] > 0
        else np.nan
    )
)

ranking["matched_query_spectra"] = (
    ranking["inchikey14"].map(matched_query_counts)
)

ranking = ranking.sort_values(
    ["spectral_score", "mass_error_ppm"],
    ascending=[False, True],
    na_position="last"
).reset_index(drop=True)

# Evaluate the known structure AFTER ranking
known_positions = np.flatnonzero(
    ranking["inchikey14"].to_numpy() == true_key
)

known_rank = (
    int(known_positions[0]) + 1
    if len(known_positions) == 1
    else None
)

print("CASMI 2026 - Spectral Ranking Experiment")
print("----------------------------------------")
print("Test molecule:", molecule_id)
print("Total candidates:", len(ranking))
print(
    "Candidates with spectral scores:",
    int(ranking["spectral_score"].notna().sum())
)
print("Known structure rank:", known_rank)
print(
    "Known structure in top 25:",
    known_rank is not None and known_rank <= 25
)

print("\nTop 5 candidates:")
print(
    ranking[
        [
            "inchikey14",
            "spectral_score",
            "matched_query_spectra",
            "mass_error_ppm"
        ]
    ].head(5).to_string(index=False)
)

if known_rank is not None:
    print("\nKnown structure:")
    print(
        ranking.loc[
            known_rank - 1,
            [
                "inchikey14",
                "spectral_score",
                "matched_query_spectra",
                "mass_error_ppm"
            ]
        ].to_string()
    )

CASMI 2026 - Spectral Ranking Experiment
----------------------------------------
Test molecule: m_006153
Total candidates: 257
Candidates with spectral scores: 250
Known structure rank: 15
Known structure in top 25: True

Top 5 candidates:
    inchikey14  spectral_score  matched_query_spectra  mass_error_ppm
OWLWDUZRCJYSHF        0.443144                      3        1.203321
JFPHIOHAVGPTKT        0.371177                      6       15.639837
SSOWPHRKXKLBRF        0.357845                      6        1.203321
VYMUBUVCGNOAMN        0.328112                      6       11.719535
MDQHIPKMJWCDLW        0.327811                      6        8.369637

Known structure:
inchikey14               BKIHGKNEJKYYAC
spectral_score                 0.246311
matched_query_spectra                 3
mass_error_ppm                 1.203321


In [14]:
# CASMI 2026 - Inspect the known candidate's spectral scores

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

known_refs = experiment_references[
    experiment_references["inchikey14"] == true_key
]

print("CASMI 2026 - Known Structure Spectral Scores")
print("---------------------------------------------")
print("Test molecule:", molecule_id)
print("Known structure:", true_key)
print("Reference spectra:", len(known_refs))

for _, query_row in queries.iterrows():

    same_adduct_refs = known_refs[
        known_refs["adduct"] == query_row["adduct"]
    ]

    if same_adduct_refs.empty:
        print(
            query_row["spectrum_id"],
            "| Adduct:", query_row["adduct"],
            "| No matching reference"
        )
        continue

    query_vector = extract_features(query_row).reshape(1, -1)

    reference_vectors = np.stack(
        same_adduct_refs.apply(
            extract_features,
            axis=1
        ).to_numpy()
    )

    best_score = cosine_similarity(
        query_vector,
        reference_vectors
    ).max()

    print(
        query_row["spectrum_id"],
        "| Adduct:", query_row["adduct"],
        "| Best similarity:", round(float(best_score), 6)
    )

CASMI 2026 - Known Structure Spectral Scores
---------------------------------------------
Test molecule: m_006153
Known structure: BKIHGKNEJKYYAC
Reference spectra: 1
s_3c74e9fe | Adduct: [M+H]+ | Best similarity: 0.148354
s_54245904 | Adduct: [M-H]- | No matching reference
s_59a80abf | Adduct: [M-H]- | No matching reference
s_7151fa86 | Adduct: [M+H]+ | Best similarity: 0.592624
s_86472581 | Adduct: [M-H]- | No matching reference
s_cb15f0ac | Adduct: [M+H]+ | Best similarity: 0.736886


In [15]:
# CASMI 2026 - Compare three spectral scoring methods

import numpy as np
import pandas as pd

comparison = ranking[
    [
        "inchikey14",
        "mass_error_ppm",
        "spectral_score",
        "matched_query_spectra"
    ]
].copy()

# Method 1: Original score
# Missing query-adduct references contribute zero.
comparison["score_original"] = comparison["spectral_score"]

# Method 2: Average over query spectra that have references.
# This does not penalize missing reference adducts.
comparison["score_available"] = comparison.apply(
    lambda row: (
        score_sums[row["inchikey14"]]
        / matched_query_counts[row["inchikey14"]]
        if matched_query_counts[row["inchikey14"]] > 0
        else np.nan
    ),
    axis=1
)

# Method 3: Average over available spectra,
# with a moderate penalty for incomplete reference coverage.
coverage = comparison["matched_query_spectra"] / len(queries)

comparison["score_coverage_adjusted"] = (
    comparison["score_available"]
    * (0.5 + 0.5 * coverage)
)

print("CASMI 2026 - Scoring Method Comparison")
print("--------------------------------------")
print("Test molecule:", molecule_id)
print("Known structure:", true_key)

for method in [
    "score_original",
    "score_available",
    "score_coverage_adjusted"
]:
    ordered = comparison.sort_values(
        [method, "mass_error_ppm"],
        ascending=[False, True],
        na_position="last"
    ).reset_index(drop=True)

    positions = np.flatnonzero(
        ordered["inchikey14"].to_numpy() == true_key
    )

    known_rank = (
        int(positions[0]) + 1
        if len(positions) == 1
        else None
    )

    print(f"\nMethod: {method}")
    print("Known structure rank:", known_rank)

    if known_rank is not None:
        print(
            "Known structure score:",
            round(
                float(ordered.loc[known_rank - 1, method]),
                6
            )
        )

    print(
        "Top candidate:",
        ordered.loc[0, "inchikey14"]
    )

CASMI 2026 - Scoring Method Comparison
--------------------------------------
Test molecule: m_006153
Known structure: BKIHGKNEJKYYAC

Method: score_original
Known structure rank: 15
Known structure score: 0.246311
Top candidate: OWLWDUZRCJYSHF

Method: score_available
Known structure rank: 4
Known structure score: 0.492621
Top candidate: OWLWDUZRCJYSHF

Method: score_coverage_adjusted
Known structure rank: 5
Known structure score: 0.369466
Top candidate: OWLWDUZRCJYSHF


In [16]:
# CASMI 2026 - Check memory before batch spectral evaluation

import os
import psutil

memory = psutil.virtual_memory()
process = psutil.Process(os.getpid())

print("CASMI 2026 - Memory Check")
print("-------------------------")
print(
    "Total RAM:",
    round(memory.total / (1024 ** 3), 2),
    "GB"
)
print(
    "Available RAM:",
    round(memory.available / (1024 ** 3), 2),
    "GB"
)
print(
    "Notebook RAM usage:",
    round(process.memory_info().rss / (1024 ** 3), 2),
    "GB"
)
print(
    "Training row groups:",
    train_file.num_row_groups
)

CASMI 2026 - Memory Check
-------------------------
Total RAM: 31.35 GB
Available RAM: 23.64 GB
Notebook RAM usage: 6.49 GB
Training row groups: 21


In [17]:
# CASMI 2026 - Prepare a reproducible evaluation subset

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

all_molecule_ids = sample_submission["molecule_id"].to_numpy()

evaluation_ids = rng.choice(
    all_molecule_ids,
    size=min(20, len(all_molecule_ids)),
    replace=False
).tolist()

# Use the same 20 ppm mass filter for every molecule
library_with_mass = full_candidate_library.dropna(
    subset=["reference_mass"]
)

reference_masses = library_with_mass["reference_mass"].to_numpy(
    dtype=float
)

evaluation_candidates = {}
all_candidate_keys = set()
all_evaluation_adducts = set()

for molecule_id in evaluation_ids:

    query_mass = float(mass_lookup.loc[molecule_id])

    ppm_errors = (
        np.abs(reference_masses - query_mass)
        / query_mass
    ) * 1_000_000

    candidates = library_with_mass.loc[
        ppm_errors <= 20
    ]

    candidate_keys = set(candidates["inchikey14"])

    evaluation_candidates[molecule_id] = candidate_keys
    all_candidate_keys.update(candidate_keys)

    molecule_adducts = set(
        test_spectra.loc[
            test_spectra["molecule_id"] == molecule_id,
            "adduct"
        ]
    )

    all_evaluation_adducts.update(molecule_adducts)


print("CASMI 2026 - Batch Evaluation Preparation")
print("-----------------------------------------")
print("Evaluation molecules:", len(evaluation_ids))

print(
    "Evaluation test spectra:",
    int(
        test_spectra["molecule_id"]
        .isin(evaluation_ids)
        .sum()
    )
)

print(
    "Unique candidate structures to search:",
    len(all_candidate_keys)
)

print(
    "Minimum candidates per molecule:",
    min(len(keys) for keys in evaluation_candidates.values())
)

print(
    "Maximum candidates per molecule:",
    max(len(keys) for keys in evaluation_candidates.values())
)

print(
    "Adducts in evaluation subset:",
    sorted(all_evaluation_adducts)
)

print("\nEvaluation molecules:")
print(evaluation_ids)

CASMI 2026 - Batch Evaluation Preparation
-----------------------------------------
Evaluation molecules: 20
Evaluation test spectra: 55
Unique candidate structures to search: 2306
Minimum candidates per molecule: 9
Maximum candidates per molecule: 347
Adducts in evaluation subset: ['[M+H]+', '[M+Na]+', '[M-H]-']

Evaluation molecules:
['m_c886af', 'm_bf6cfe', 'm_30024e', 'm_f866a0', 'm_e138be', 'm_c9a4bc', 'm_1de28b', 'm_153108', 'm_acdfec', 'm_8d19fa', 'm_7108be', 'm_145679', 'm_7b993b', 'm_dc7be7', 'm_15dab4', 'm_c3e52e', 'm_d07b86', 'm_741f2c', 'm_b8f328', 'm_8d75c7']


In [18]:
# CASMI 2026 - Small, memory-safe reference search test

import os
import psutil

process = psutil.Process(os.getpid())

memory_before = process.memory_info().rss / (1024 ** 3)

# Read only two metadata columns from the first 2,000 rows
small_batch = next(
    train_file.iter_batches(
        batch_size=2000,
        columns=["inchikey14", "adduct"]
    )
).to_pandas()

# Find candidate structures from our 20-molecule evaluation set
matches = small_batch[
    small_batch["inchikey14"].isin(all_candidate_keys)
    & small_batch["adduct"].isin(all_evaluation_adducts)
]

memory_after = process.memory_info().rss / (1024 ** 3)

print("CASMI 2026 - Safe Reference Search Test")
print("---------------------------------------")
print("Rows inspected:", len(small_batch))
print("Matching rows:", len(matches))
print("Unique matching structures:", matches["inchikey14"].nunique())
print("Notebook RAM before:", round(memory_before, 3), "GB")
print("Notebook RAM after:", round(memory_after, 3), "GB")
print("RAM change:", round(memory_after - memory_before, 3), "GB")

CASMI 2026 - Safe Reference Search Test
---------------------------------------
Rows inspected: 2000
Matching rows: 2
Unique matching structures: 2
Notebook RAM before: 6.494 GB
Notebook RAM after: 6.495 GB
RAM change: 0.0 GB


In [19]:
# CASMI 2026 - Bounded reference collection test

import os
import psutil
import pandas as pd

process = psutil.Process(os.getpid())

MAX_ROWS = 50000
MAX_REFERENCES = 500
BATCH_SIZE = 5000

memory_before = process.memory_info().rss / (1024 ** 3)

pilot_parts = []
rows_inspected = 0
references_collected = 0

columns = [
    "inchikey14",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

# Read a maximum of 10 batches from row group 0
for batch in train_file.iter_batches(
    row_groups=[0],
    batch_size=BATCH_SIZE,
    columns=columns
):
    batch_df = batch.to_pandas()

    rows_inspected += len(batch_df)

    matches = batch_df[
        batch_df["inchikey14"].isin(all_candidate_keys)
        & batch_df["adduct"].isin(all_evaluation_adducts)
    ]

    # Do not collect more than 500 reference spectra
    remaining_capacity = (
        MAX_REFERENCES - references_collected
    )

    if remaining_capacity > 0 and not matches.empty:
        selected = matches.head(remaining_capacity).copy()
        pilot_parts.append(selected)

        references_collected += len(selected)

    if (
        rows_inspected >= MAX_ROWS
        or references_collected >= MAX_REFERENCES
    ):
        break

pilot_references = (
    pd.concat(pilot_parts, ignore_index=True)
    if pilot_parts
    else pd.DataFrame(columns=columns)
)

memory_after = process.memory_info().rss / (1024 ** 3)

print("CASMI 2026 - Bounded Reference Collection")
print("-----------------------------------------")
print("Rows inspected:", rows_inspected)
print("Reference spectra collected:", len(pilot_references))
print(
    "Unique candidate structures:",
    pilot_references["inchikey14"].nunique()
)

print(
    "RAM before:",
    round(memory_before, 3),
    "GB"
)

print(
    "RAM after:",
    round(memory_after, 3),
    "GB"
)

print(
    "RAM change:",
    round(memory_after - memory_before, 3),
    "GB"
)

print(
    "Available RAM:",
    round(
        psutil.virtual_memory().available / (1024 ** 3),
        3
    ),
    "GB"
)

# Free the temporary references after measuring memory
del pilot_references
del pilot_parts
del batch_df

print("\nBounded collection test completed!")

CASMI 2026 - Bounded Reference Collection
-----------------------------------------
Rows inspected: 50000
Reference spectra collected: 379
Unique candidate structures: 92
RAM before: 6.495 GB
RAM after: 6.727 GB
RAM change: 0.232 GB
Available RAM: 23.409 GB

Bounded collection test completed!


In [20]:
# CASMI 2026 - Bounded reference collection test

import os
import psutil
import pandas as pd

process = psutil.Process(os.getpid())

MAX_ROWS = 50000
MAX_REFERENCES = 500
BATCH_SIZE = 5000

memory_before = process.memory_info().rss / (1024 ** 3)

pilot_parts = []
rows_inspected = 0
references_collected = 0

columns = [
    "inchikey14",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

# Read a maximum of 10 batches from row group 0
for batch in train_file.iter_batches(
    row_groups=[0],
    batch_size=BATCH_SIZE,
    columns=columns
):
    batch_df = batch.to_pandas()

    rows_inspected += len(batch_df)

    matches = batch_df[
        batch_df["inchikey14"].isin(all_candidate_keys)
        & batch_df["adduct"].isin(all_evaluation_adducts)
    ]

    # Do not collect more than 500 reference spectra
    remaining_capacity = (
        MAX_REFERENCES - references_collected
    )

    if remaining_capacity > 0 and not matches.empty:
        selected = matches.head(remaining_capacity).copy()
        pilot_parts.append(selected)

        references_collected += len(selected)

    if (
        rows_inspected >= MAX_ROWS
        or references_collected >= MAX_REFERENCES
    ):
        break

pilot_references = (
    pd.concat(pilot_parts, ignore_index=True)
    if pilot_parts
    else pd.DataFrame(columns=columns)
)

memory_after = process.memory_info().rss / (1024 ** 3)

print("CASMI 2026 - Bounded Reference Collection")
print("-----------------------------------------")
print("Rows inspected:", rows_inspected)
print("Reference spectra collected:", len(pilot_references))
print(
    "Unique candidate structures:",
    pilot_references["inchikey14"].nunique()
)

print(
    "RAM before:",
    round(memory_before, 3),
    "GB"
)

print(
    "RAM after:",
    round(memory_after, 3),
    "GB"
)

print(
    "RAM change:",
    round(memory_after - memory_before, 3),
    "GB"
)

print(
    "Available RAM:",
    round(
        psutil.virtual_memory().available / (1024 ** 3),
        3
    ),
    "GB"
)

# Free the temporary references after measuring memory
del pilot_references
del pilot_parts
del batch_df

print("\nBounded collection test completed!")

CASMI 2026 - Bounded Reference Collection
-----------------------------------------
Rows inspected: 50000
Reference spectra collected: 379
Unique candidate structures: 92
RAM before: 6.727 GB
RAM after: 6.696 GB
RAM change: -0.031 GB
Available RAM: 23.43 GB

Bounded collection test completed!


In [21]:
# CASMI 2026 - Memory-limited reference collection

import os
import gc
import psutil
import pandas as pd

process = psutil.Process(os.getpid())

BATCH_SIZE = 5000
MAX_REFERENCES = 40000
MAX_PROCESS_RAM_GB = 12.0
MIN_AVAILABLE_RAM_GB = 6.0

reference_columns = [
    "inchikey14",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

# Fingerprints of all test spectra in our evaluation subset
evaluation_queries = test_spectra[
    test_spectra["molecule_id"].isin(evaluation_ids)
]

evaluation_fingerprints = {
    spectrum_fingerprint(row)
    for row in evaluation_queries.itertuples(index=False)
}

reference_parts = []
references_collected = 0
exact_copies_excluded = 0
rows_inspected = 0

collection_complete = True
stop_reason = None

print("CASMI 2026 - Reference Collection")
print("---------------------------------")

for group_index in range(train_file.num_row_groups):

    memory = psutil.virtual_memory()
    ram_used = process.memory_info().rss / (1024 ** 3)
    ram_available = memory.available / (1024 ** 3)

    # Stop before reading another row group if memory is low
    if (
        ram_used >= MAX_PROCESS_RAM_GB
        or ram_available <= MIN_AVAILABLE_RAM_GB
    ):
        collection_complete = False
        stop_reason = "RAM safety limit reached"
        break

    for batch in train_file.iter_batches(
        row_groups=[group_index],
        batch_size=BATCH_SIZE,
        columns=reference_columns
    ):
        batch_df = batch.to_pandas()
        rows_inspected += len(batch_df)

        matches = batch_df[
            batch_df["inchikey14"].isin(all_candidate_keys)
            & batch_df["adduct"].isin(all_evaluation_adducts)
        ].copy()

        if not matches.empty:

            # Remove exact copies of evaluation test spectra
            is_copy = [
                spectrum_fingerprint(row)
                in evaluation_fingerprints
                for row in matches.itertuples(index=False)
            ]

            exact_copies_excluded += sum(is_copy)

            remaining = matches.loc[
                ~pd.Series(is_copy, index=matches.index)
            ]

            if not remaining.empty:
                reference_parts.append(remaining)
                references_collected += len(remaining)

        del batch_df, matches

        # Check memory after each batch
        ram_used = process.memory_info().rss / (1024 ** 3)
        ram_available = (
            psutil.virtual_memory().available / (1024 ** 3)
        )

        if references_collected >= MAX_REFERENCES:
            collection_complete = False
            stop_reason = "Reference count limit reached"
            break

        if (
            ram_used >= MAX_PROCESS_RAM_GB
            or ram_available <= MIN_AVAILABLE_RAM_GB
        ):
            collection_complete = False
            stop_reason = "RAM safety limit reached"
            break

    print(
        f"Row groups checked: {group_index + 1}/"
        f"{train_file.num_row_groups} | "
        f"References: {references_collected} | "
        f"RAM: {ram_used:.2f} GB"
    )

    gc.collect()

    if not collection_complete:
        break


evaluation_references = (
    pd.concat(reference_parts, ignore_index=True)
    if reference_parts
    else pd.DataFrame(columns=reference_columns)
)

del reference_parts
gc.collect()

print("\nCASMI 2026 - Collection Results")
print("--------------------------------")
print("Collection complete:", collection_complete)
print("Rows inspected:", rows_inspected)
print("References collected:", len(evaluation_references))
print(
    "Unique reference structures:",
    evaluation_references["inchikey14"].nunique()
)
print("Exact copies excluded:", exact_copies_excluded)

print(
    "Current notebook RAM:",
    round(process.memory_info().rss / (1024 ** 3), 2),
    "GB"
)

if not collection_complete:
    print("Stopped early:", stop_reason)
else:
    print("All training row groups checked successfully!")

CASMI 2026 - Reference Collection
---------------------------------
Row groups checked: 1/21 | References: 997 | RAM: 6.99 GB
Row groups checked: 2/21 | References: 1920 | RAM: 7.44 GB
Row groups checked: 3/21 | References: 2791 | RAM: 7.94 GB
Row groups checked: 4/21 | References: 3767 | RAM: 8.41 GB
Row groups checked: 5/21 | References: 4713 | RAM: 8.88 GB
Row groups checked: 6/21 | References: 5468 | RAM: 9.41 GB
Row groups checked: 7/21 | References: 6380 | RAM: 9.88 GB
Row groups checked: 8/21 | References: 7281 | RAM: 10.39 GB
Row groups checked: 9/21 | References: 7987 | RAM: 10.85 GB
Row groups checked: 10/21 | References: 8282 | RAM: 11.40 GB
Row groups checked: 11/21 | References: 8531 | RAM: 11.50 GB
Row groups checked: 12/21 | References: 8742 | RAM: 11.91 GB
Row groups checked: 13/21 | References: 9234 | RAM: 11.95 GB
Row groups checked: 14/21 | References: 10056 | RAM: 11.79 GB
Row groups checked: 15/21 | References: 10847 | RAM: 11.87 GB
Row groups checked: 16/21 | Refe

In [22]:
# CASMI 2026 - Convert collected references to compact feature vectors

import gc
import os
import numpy as np
import psutil

process = psutil.Process(os.getpid())

ram_before = process.memory_info().rss / (1024 ** 3)

# Keep only the metadata needed for candidate ranking
reference_metadata = evaluation_references[
    ["inchikey14", "adduct"]
].copy().reset_index(drop=True)

reference_count = len(evaluation_references)

# Allocate a compact float32 matrix
reference_features = np.zeros(
    (reference_count, 1000),
    dtype=np.float32
)

# Convert one reference spectrum at a time
for i, row in enumerate(
    evaluation_references.itertuples(index=False)
):

    mz = np.asarray(row.ms2_mzs, dtype=float)

    intensity = np.asarray(
        row.ms2_normalized_intensities,
        dtype=float
    )

    valid = (
        np.isfinite(mz)
        & np.isfinite(intensity)
        & (mz >= 0)
        & (mz < 1000)
    )

    bins = mz[valid].astype(np.int32)

    # Retain the maximum intensity in each 1 Da bin
    np.maximum.at(
        reference_features[i],
        bins,
        intensity[valid].astype(np.float32)
    )

# Release the large DataFrame containing raw fragment arrays
del evaluation_references
gc.collect()

ram_after = process.memory_info().rss / (1024 ** 3)

print("CASMI 2026 - Compact Reference Features")
print("---------------------------------------")
print("Reference spectra converted:", reference_count)
print("Feature matrix shape:", reference_features.shape)

print(
    "Feature matrix size:",
    round(reference_features.nbytes / (1024 ** 2), 2),
    "MB"
)

print(
    "Metadata rows:",
    len(reference_metadata)
)

print("RAM before:", round(ram_before, 2), "GB")
print("RAM after:", round(ram_after, 2), "GB")

print("\nCompact reference features prepared!")

CASMI 2026 - Compact Reference Features
---------------------------------------
Reference spectra converted: 11798
Feature matrix shape: (11798, 1000)
Feature matrix size: 45.01 MB
Metadata rows: 11798
RAM before: 12.0 GB
RAM after: 12.05 GB

Compact reference features prepared!
